# Model validation: does the 88% hold up?

The main notebook trains a Decision Tree on a random 80/20 split and reaches **88% accuracy**. The dataset has many identical rows, so this notebook checks whether those duplicates make the score look better than it is.

In [1]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
from scripts.build import load_clean, model_checks

df = load_clean()
profile = df.drop(columns='Person ID').astype(str).agg('|'.join, axis=1)
print(f'{len(df)} rows, {profile.nunique()} unique profiles, {len(df) - profile.nunique()} rows repeat an earlier profile')

374 rows, 132 unique profiles, 242 rows repeat an earlier profile


## 1. How many test rows were already seen in training?

In [2]:
r = model_checks(df)
o = r['original_split']
print(f"Original split: accuracy {o['accuracy']:.1%} on {o['test_rows']} test rows")
print(f"{o['test_rows_also_in_training']} of the {o['test_rows']} test rows are identical to a training row")

Original split: accuracy 88.0% on 75 test rows
66 of the 75 test rows are identical to a training row


Most test rows have an exact copy in the training data, so a random split mostly tests the model on people it has already seen.

In [3]:
pd.DataFrame(o['confusion_matrix'], index=[f'actual {c}' for c in r['classes']], columns=[f'predicted {c}' for c in r['classes']])

,predicted Insomnia,predicted None,predicted Sleep Apnea
actual Insomnia,15,2,3
actual None,1,40,0
actual Sleep Apnea,2,1,11


## 2. Leak-free test: keep identical profiles together

`GroupKFold` puts every copy of a profile in the same fold, so the model is always tested on profiles it has never seen.

In [4]:
g = r['group_kfold']
print('Fold accuracies:', g['folds'])
print(f"Mean accuracy: {g['mean_accuracy']:.1%}")

Fold accuracies: [0.9467, 0.88, 0.92, 0.8533, 0.8649]
Mean accuracy: 89.3%


## 3. Each profile counted once

Dropping duplicates leaves one row per profile. This weights rare profiles as much as common ones.

In [5]:
u = r['unique_profiles_cv']
print(f"5-fold accuracy on {u['rows']} unique profiles: {u['mean_accuracy']:.1%}")
print(f"Baseline (always predict the most common class): {r['unique_profiles_baseline']:.1%}")

5-fold accuracy on 132 unique profiles: 63.6%
Baseline (always predict the most common class): 55.3%


## 4. What the model relies on

In [6]:
pd.DataFrame(r['feature_importance'])

,feature,importance
0,BP Systolic,0.4758
1,BMI Category,0.3913
2,Daily Steps,0.1118
3,Physical Activity Level,0.0182
4,Age,0.0016
5,Sleep Duration,0.0012


## Conclusion

- On new profiles drawn the way the data is (`GroupKFold`), the model still reaches about **89%**, so the original **88%** is not mainly caused by duplicate rows.
- When every unique profile counts once, accuracy falls to about **64%**, against a 55% baseline. The common, repeated profiles are the easy ones; unusual profiles are much harder to classify.
- The model relies mostly on **systolic blood pressure** and **BMI category**. The dataset is synthetic, so these patterns show how the data was generated, not medical cause and effect.